# Project 4 - NOAA Weather Data Analysis with PySpark

Analyzing NCEI Global Surface Summary of Day data for Cincinnati (72429793812) and Florida (99495199999) from 2015-2024.

## Setup - Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType
import os

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Project4_WeatherAnalysis") \
    .master("local[*]") \
    .getOrCreate()

spark

## Question 2 - Load CSV files and display the count of each dataset
There should be 19 results (no Florida data for 2016).

In [ ]:
# Define base path and station info
base_path = "noaa_data"
years = list(range(2015, 2025))
stations = {
    "72429793812": "Cincinnati",
    "99495199999": "Florida"
}

# Load all CSV files and display counts
dataframes = {}
for year in years:
    for station_id, station_name in stations.items():
        file_path = os.path.join(base_path, str(year), f"{station_id}.csv")
        if os.path.exists(file_path):
            df = spark.read.csv(file_path, header=True, inferSchema=True)
            dataframes[(year, station_id)] = df
            print(f"{year} - {station_name} ({station_id}): {df.count()} records")
        else:
            print(f"{year} - {station_name} ({station_id}): FILE DOES NOT EXIST")

print(f"\nTotal datasets loaded: {len(dataframes)}")

In [ ]:
# Combine all dataframes into one for subsequent analysis
from functools import reduce
from pyspark.sql import DataFrame

all_data = reduce(DataFrame.unionByName, dataframes.values())

# Clean the data: cast numeric columns and handle missing values
# Missing value indicators per README: 9999.9 for TEMP/DEWP/MAX/MIN, 999.9 for VISIB/WDSP/GUST/SNDP, 99.99 for PRCP
all_data = all_data \
    .withColumn("TEMP", F.col("TEMP").cast(DoubleType())) \
    .withColumn("MAX", F.col("MAX").cast(DoubleType())) \
    .withColumn("MIN", F.col("MIN").cast(DoubleType())) \
    .withColumn("PRCP", F.col("PRCP").cast(DoubleType())) \
    .withColumn("GUST", F.col("GUST").cast(DoubleType())) \
    .withColumn("WDSP", F.col("WDSP").cast(DoubleType())) \
    .withColumn("FRSHTT", F.col("FRSHTT").cast(StringType())) \
    .withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd")) \
    .withColumn("YEAR", F.year(F.col("DATE"))) \
    .withColumn("MONTH", F.month(F.col("DATE")))

all_data.printSchema()
print(f"Total combined records: {all_data.count()}")

## Question 3 - Find the hottest day (MAX) for each year
Show STATION, NAME, DATE, MAX. There should be 10 results (one per year).

In [ ]:
# Filter out missing MAX values (9999.9)
valid_max = all_data.filter(F.col("MAX") < 9999.9)

# Find the hottest day per year using window function
window_max = Window.partitionBy("YEAR").orderBy(F.col("MAX").desc())
hottest_days = valid_max \
    .withColumn("rank", F.row_number().over(window_max)) \
    .filter(F.col("rank") == 1) \
    .select("STATION", "NAME", "DATE", "MAX") \
    .orderBy("DATE")

hottest_days.show(10, truncate=False)

## Question 4 - Find the coldest day (MIN) for March across all years (2015-2024)
Show STATION, NAME, DATE, MIN. There should be 1 result.

In [ ]:
# Filter for March and valid MIN values
march_data = all_data.filter(
    (F.col("MONTH") == 3) & (F.col("MIN") < 9999.9)
)

# Find the single coldest day in March across all years
coldest_march = march_data \
    .orderBy(F.col("MIN").asc()) \
    .select("STATION", "NAME", "DATE", "MIN") \
    .limit(1)

coldest_march.show(truncate=False)

## Question 5 - Find the year with the most precipitation for Cincinnati and Florida
Show STATION, NAME, YEAR, Mean of PRCP. There should be 2 results.

In [ ]:
# Filter out missing PRCP values (99.99)
valid_prcp = all_data.filter(F.col("PRCP") < 99.99)

# Calculate mean precipitation per station per year
prcp_by_year = valid_prcp.groupBy("STATION", "NAME", "YEAR") \
    .agg(F.mean("PRCP").alias("Mean of PRCP"))

# Find the year with highest mean precipitation for each station
window_prcp = Window.partitionBy("STATION").orderBy(F.col("Mean of PRCP").desc())
most_prcp = prcp_by_year \
    .withColumn("rank", F.row_number().over(window_prcp)) \
    .filter(F.col("rank") == 1) \
    .select("STATION", "NAME", "YEAR", "Mean of PRCP") \
    .orderBy("STATION")

most_prcp.show(truncate=False)

## Question 6 - Percentage of missing values for GUST in 2024
For Cincinnati and Florida. There should be 2 results.

In [ ]:
# Filter for 2024 data
data_2024 = all_data.filter(F.col("YEAR") == 2024)

# Calculate percentage of missing GUST values (999.9) per station
gust_missing = data_2024.groupBy("STATION", "NAME") \
    .agg(
        F.count("*").alias("total_records"),
        F.sum(F.when(F.col("GUST") >= 999.9, 1).otherwise(0)).alias("missing_count")
    ) \
    .withColumn("missing_percentage", F.round((F.col("missing_count") / F.col("total_records")) * 100, 2)) \
    .select("STATION", "NAME", "total_records", "missing_count", "missing_percentage") \
    .orderBy("STATION")

gust_missing.show(truncate=False)

## Question 7 - Mean, Median, Mode, and Std Dev of TEMP for Cincinnati by month in 2020
There should be 12 results (one per month, 4 values each).

In [ ]:
# Filter for Cincinnati 2020 with valid TEMP
cincy_2020 = all_data.filter(
    (F.col("STATION") == 72429793812) &
    (F.col("YEAR") == 2020) &
    (F.col("TEMP") < 9999.9)
)

# Calculate mean and stddev per month
stats_by_month = cincy_2020.groupBy("MONTH") \
    .agg(
        F.round(F.mean("TEMP"), 2).alias("Mean"),
        F.round(F.percentile_approx("TEMP", 0.5), 2).alias("Median"),
        F.round(F.stddev("TEMP"), 2).alias("Std_Dev")
    ) \
    .orderBy("MONTH")

# Calculate mode separately (most frequent TEMP value per month)
temp_counts = cincy_2020.groupBy("MONTH", "TEMP").count()
window_mode = Window.partitionBy("MONTH").orderBy(F.col("count").desc(), F.col("TEMP").asc())
mode_by_month = temp_counts \
    .withColumn("rank", F.row_number().over(window_mode)) \
    .filter(F.col("rank") == 1) \
    .select(F.col("MONTH"), F.col("TEMP").alias("Mode"))

# Join mode with other stats
result_q7 = stats_by_month.join(mode_by_month, on="MONTH") \
    .select("MONTH", "Mean", "Median", "Mode", "Std_Dev") \
    .orderBy("MONTH")

result_q7.show(12, truncate=False)

## Question 8 - Top 10 days with lowest Wind Chill for Cincinnati in 2017
Filter: TEMP < 50°F and WDSP > 3 mph

Wind Chill formula: WC = 35.74 + 0.6215 × TEMP − 35.75 × (WDSP)^0.16 + 0.4275 × TEMP × (WDSP)^0.16

In [ ]:
# Filter Cincinnati 2017, TEMP < 50, WDSP > 3 mph, valid values
# WDSP is recorded in knots; convert to mph by multiplying by 1.15078
cincy_2017 = all_data.filter(
    (F.col("STATION") == 72429793812) &
    (F.col("YEAR") == 2017) &
    (F.col("TEMP") < 50) &
    (F.col("TEMP") < 9999.9) &
    ((F.col("WDSP") * 1.15078) > 3) &
    (F.col("WDSP") < 999.9)
)

# Calculate Wind Chill (WDSP converted from knots to mph)
# WC = 35.74 + 0.6215 * TEMP - 35.75 * (WDSP_mph)^0.16 + 0.4275 * TEMP * (WDSP_mph)^0.16
wind_chill = cincy_2017.withColumn(
    "Wind_Chill",
    F.round(
        35.74 + 0.6215 * F.col("TEMP") 
        - 35.75 * F.pow((F.col("WDSP") * 1.15078), 0.16) 
        + 0.4275 * F.col("TEMP") * F.pow((F.col("WDSP") * 1.15078), 0.16),
        2
    )
)

# Display top 10 lowest wind chill days
top10_wc = wind_chill \
    .select("STATION", "NAME", "DATE", "TEMP", "WDSP", "Wind_Chill") \
    .orderBy(F.col("Wind_Chill").asc()) \
    .limit(10)

top10_wc.show(10, truncate=False)

## Question 9 - Days with extreme weather conditions for Florida (FRSHTT)
FRSHTT is a 6-digit string where each digit indicates: Fog, Rain, Snow, Hail, Thunder, Tornado.
Count days where any extreme condition occurred. There should be 1 result.

In [ ]:
# Filter for Florida station
florida_data = all_data.filter(F.col("STATION") == 99495199999)

# Pad FRSHTT to 6 characters (in case of leading zeros lost)
florida_frshtt = florida_data.withColumn(
    "FRSHTT_padded", F.lpad(F.col("FRSHTT"), 6, "0")
)

# A day has extreme weather if any digit in FRSHTT is '1'
extreme_days = florida_frshtt.filter(F.col("FRSHTT_padded") != "000000")

# Count total extreme weather days
extreme_count = extreme_days.count()
print(f"Total days with extreme weather conditions for Florida: {extreme_count}")

# Also break down by type
breakdown = florida_frshtt.select(
    F.sum(F.when(F.substring("FRSHTT_padded", 1, 1) == "1", 1).otherwise(0)).alias("Fog"),
    F.sum(F.when(F.substring("FRSHTT_padded", 2, 1) == "1", 1).otherwise(0)).alias("Rain"),
    F.sum(F.when(F.substring("FRSHTT_padded", 3, 1) == "1", 1).otherwise(0)).alias("Snow"),
    F.sum(F.when(F.substring("FRSHTT_padded", 4, 1) == "1", 1).otherwise(0)).alias("Hail"),
    F.sum(F.when(F.substring("FRSHTT_padded", 5, 1) == "1", 1).otherwise(0)).alias("Thunder"),
    F.sum(F.when(F.substring("FRSHTT_padded", 6, 1) == "1", 1).otherwise(0)).alias("Tornado")
)
breakdown.show(truncate=False)

## Question 10 - Predict MAX Temperature for Cincinnati for Nov & Dec 2024
Based on previous 2 years of weather data (2022-2023). Using Linear Regression.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Get Cincinnati data for 2022-2023 (training) with valid MAX
cincy_train = all_data.filter(
    (F.col("STATION") == 72429793812) &
    (F.col("YEAR").isin(2022, 2023)) &
    (F.col("MAX") < 9999.9) &
    (F.col("TEMP") < 9999.9) &
    (F.col("MIN") < 9999.9)
)

# Features: MONTH, day of year, TEMP, MIN
cincy_train = cincy_train.withColumn("DAY_OF_YEAR", F.dayofyear(F.col("DATE")))

# Prepare feature vector
assembler = VectorAssembler(
    inputCols=["MONTH", "DAY_OF_YEAR", "TEMP", "MIN"],
    outputCol="features"
)
train_data = assembler.transform(cincy_train).select("features", F.col("MAX").alias("label"), "DATE", "MONTH", "YEAR")

# Train Linear Regression model
lr = LinearRegression(maxIter=100, regParam=0.01, elasticNetParam=0.8)
lr_model = lr.fit(train_data)

print(f"Model Coefficients: {lr_model.coefficients}")
print(f"Model Intercept: {lr_model.intercept}")
print(f"Training RMSE: {lr_model.summary.rootMeanSquaredError:.2f}")
print(f"Training R2: {lr_model.summary.r2:.4f}")

In [ ]:
# Get actual Nov & Dec 2024 data for Cincinnati to use as test input
cincy_nov_dec_2024 = all_data.filter(
    (F.col("STATION") == 72429793812) &
    (F.col("YEAR") == 2024) &
    (F.col("MONTH").isin(11, 12)) &
    (F.col("MAX") < 9999.9) &
    (F.col("TEMP") < 9999.9) &
    (F.col("MIN") < 9999.9)
).withColumn("DAY_OF_YEAR", F.dayofyear(F.col("DATE")))

test_data = assembler.transform(cincy_nov_dec_2024).select("features", F.col("MAX").alias("label"), "DATE", "MONTH")

# Make predictions
predictions = lr_model.transform(test_data)

# Calculate predicted MAX for Nov and Dec
predicted_max = predictions.groupBy("MONTH") \
    .agg(
        F.round(F.max("prediction"), 2).alias("Predicted_MAX"),
        F.round(F.max("label"), 2).alias("Actual_MAX")
    ) \
    .orderBy("MONTH")

print("Predicted vs Actual MAX Temperature for Cincinnati Nov & Dec 2024:")
predicted_max.show(truncate=False)

# Evaluate model on test data
evaluator_rmse = RegressionEvaluator(metricName="rmse")
evaluator_r2 = RegressionEvaluator(metricName="r2")
test_rmse = evaluator_rmse.evaluate(predictions)
test_r2 = evaluator_r2.evaluate(predictions)
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Test R2: {test_r2:.4f}")

### Model Discussion

**Model Used:** Linear Regression with features: MONTH, DAY_OF_YEAR, TEMP (mean daily temperature), and MIN (minimum temperature).

**Performance:** The model leverages the strong correlation between daily mean/min temperatures and max temperature. The R² score indicates how well the model explains variance in MAX temperature.

**Potential Improvements:**
- Use additional features such as DEWP (dew point), SLP (sea level pressure), and WDSP (wind speed)
- Try non-linear models like Random Forest or Gradient Boosted Trees
- Include more years of historical data for training
- Add lag features (previous day's MAX) to capture temporal patterns
- Use time-series specific models like ARIMA for better seasonal pattern capture

In [ ]:
# Stop Spark session
spark.stop()